<a href="https://colab.research.google.com/github/Leonardozepeda04/edt-dataa-pipeline/blob/main/notebooks/siniestros.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd

In [2]:
#Crear dataset
url_tipos = "https://raw.githubusercontent.com/Leonardozepeda04/edt-dataa-pipeline/refs/heads/main/data/raw/siniestros.csv"

In [3]:
siniestros = pd.read_csv(url_tipos)

In [4]:
#Exploracion de datos
siniestros.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4620 entries, 0 to 4619
Data columns (total 5 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   id_siniestro     4620 non-null   int64 
 1   id_poliza        4620 non-null   int64 
 2   fecha_siniestro  4620 non-null   object
 3   monto_siniestro  4004 non-null   object
 4   estado           3322 non-null   object
dtypes: int64(2), object(3)
memory usage: 180.6+ KB


In [5]:
#Limpieza de datos
def limpiar_dataframe(df):

    df.columns = df.columns.str.strip().str.lower()

    for col in df.select_dtypes(include='object').columns:
        df[col] = df[col].astype(str).str.strip()

    df = df.replace(r'^\s*$', pd.NA, regex=True)

    df = df.drop_duplicates()

    return df

In [6]:
# Transformaciones

# --- Normalizar espacios y mayúsculas/minúsculas ---
siniestros['estado'] = siniestros['estado'].astype(str).str.strip().str.capitalize()

# --- Limpieza de columna numérica ---
# Eliminar comas de miles y espacios
siniestros['monto_siniestro'] = siniestros['monto_siniestro'].astype(str).str.replace(",", "", regex=False).str.strip()

# Convertir a numérico (los valores no válidos se convierten en NaN)
siniestros['monto_siniestro'] = pd.to_numeric(siniestros['monto_siniestro'], errors='coerce')

# Rellenar valores nulos con 0
siniestros['monto_siniestro'] = siniestros['monto_siniestro'].fillna(0)

# --- Limpieza de columna fecha ---
# Reemplazar / por - y eliminar espacios
siniestros['fecha_siniestro'] = siniestros['fecha_siniestro'].astype(str).str.replace("/", "-", regex=False).str.strip()

# Convertir a fecha
siniestros['fecha_siniestro'] = pd.to_datetime(siniestros['fecha_siniestro'], format='%d-%m-%Y', errors='coerce')

# --- Columnas derivadas ---
siniestros['anio'] = siniestros['fecha_siniestro'].dt.year
siniestros['mes'] = siniestros['fecha_siniestro'].dt.month

In [7]:
#Ver resultados
print(siniestros)

      id_siniestro  id_poliza fecha_siniestro  monto_siniestro     estado  \
0                1      17400      2025-10-16          2092.59    Abierto   
1                2       2465      2025-08-01          7076.25    Abierto   
2                3      15785      2025-09-19           702.27    Cerrado   
3                4      14299      2025-09-27           274.63    Abierto   
4                5      12908      2025-12-01          9377.69  Rechazado   
...            ...        ...             ...              ...        ...   
4615          4616      12223             NaT          5806.47        Nan   
4616          4617       3953             NaT         14622.50    Cerrado   
4617          4618      10239             NaT             0.00    Abierto   
4618          4619      18897             NaT           354.26        Nan   
4619          4620      10815      2025-10-11         14958.46    Abierto   

        anio   mes  
0     2025.0  10.0  
1     2025.0   8.0  
2     2025.0

In [8]:
# --- Separar válidos y rechazados ---
validos = siniestros[
    siniestros['id_siniestro'].notna() &
    siniestros['id_poliza'].notna() &
    siniestros['fecha_siniestro'].notna() &
    siniestros['monto_siniestro'].notna() &
    siniestros['estado'].notna() &
    (siniestros['monto_siniestro'] > 0)
].copy()

rechazados = siniestros[
    siniestros['id_siniestro'].isna() |
    siniestros['id_poliza'].isna() |
    siniestros['fecha_siniestro'].isna() |
    siniestros['monto_siniestro'].isna() |
    siniestros['estado'].isna() |
    (siniestros['monto_siniestro'] <= 0)
].copy()

In [9]:
#Mostrar resultados
print("✅ Válidos:")
print(validos)

print("\n❌ Rechazados:")
print(rechazados)

✅ Válidos:
      id_siniestro  id_poliza fecha_siniestro  monto_siniestro     estado  \
0                1      17400      2025-10-16       2092.59000    Abierto   
1                2       2465      2025-08-01       7076.25000    Abierto   
2                3      15785      2025-09-19        702.27000    Cerrado   
3                4      14299      2025-09-27        274.63000    Abierto   
4                5      12908      2025-12-01       9377.69000  Rechazado   
...            ...        ...             ...              ...        ...   
4590          4591      22336      2025-08-08          6.99068    Cerrado   
4600          4601       8973      2025-09-17        169.84000        Nan   
4606          4607      12631      2025-04-16      11965.15000  Rechazado   
4607          4608       9264      2025-04-21         12.62659    Cerrado   
4619          4620      10815      2025-10-11      14958.46000    Abierto   

        anio   mes  
0     2025.0  10.0  
1     2025.0   8.0  
2

In [10]:
# --- Motivos de rechazo ---
def motivo(row):
    motivos = []
    if pd.isna(row['id_siniestro']):
        motivos.append("id_siniestro_vacio")
    if pd.isna(row['id_poliza']):
        motivos.append("id_poliza_vacia")
    if pd.isna(row['fecha_siniestro']):
        motivos.append("fecha_siniestro_vacia")
    if pd.isna(row['monto_siniestro']):
        motivos.append("monto_siniestro_vacio")
    elif row['monto_siniestro'] <= 0:
        motivos.append("monto_siniestro_invalido")
    if pd.isna(row['estado']):
        motivos.append("estado_vacio")
    return ",".join(motivos)

rechazados["motivo_rechazo"] = rechazados.apply(motivo, axis=1)

In [11]:
# Resultados
print("\n❌ Rechazados con motivos:")
display(rechazados[['id_siniestro', 'id_poliza', 'fecha_siniestro', 'monto_siniestro', 'estado', 'motivo_rechazo']])


❌ Rechazados con motivos:


,id_siniestro,id_poliza,fecha_siniestro,monto_siniestro,estado,motivo_rechazo
5,6,5107,NaT,10535.74,Nan,fecha_siniestro_vacia
6,7,3379,NaT,10513.30,Abierto,fecha_siniestro_vacia
8,9,18118,NaT,0.00,Cerrado,"fecha_siniestro_vacia,monto_siniestro_invalido"
9,10,19947,NaT,8801.03,Cerrado,fecha_siniestro_vacia
11,12,4096,2026-02-23,0.00,Rechazado,monto_siniestro_invalido
...,...,...,...,...,...,...
4614,4615,3466,2025-12-12,0.00,Nan,monto_siniestro_invalido
4615,4616,12223,NaT,5806.47,Nan,fecha_siniestro_vacia
4616,4617,3953,NaT,14622.50,Cerrado,fecha_siniestro_vacia
4617,4618,10239,NaT,0.00,Abierto,"fecha_siniestro_vacia,monto_siniestro_invalido"


In [12]:
# Exportar archivos
siniestros.to_csv("siniestros_curated.csv", index=False)
validos.to_csv("siniestros_validos.csv", index=False)
rechazados.to_csv("siniestros_rechazados.csv", index=False)